<a href="https://colab.research.google.com/github/Emmanuel-afk/python-random-quote/blob/master/assignment_ii_revised_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K6312 Assignment II: IMDB Review Sentiment Analysis

In this assignment, we will explore one dataset for binary sentiment classification. The original dataset contains 50,000 reviews — 25,000 positive and 25,000 negative reviews.

The number of stars would be a good proxy for sentiment classification. For example, we could pre-assign the following:
1. At least 7 out of 10 stars => positive (label=1)
2. At most 4 out of 10 stars => negative (label=0)

Here, we sample the original dataset and create a small-size data for you.

In [1]:
import pandas as pd, glob
train_path = glob.glob("/content/**/train_IMDB.csv", recursive=True)[0]
test_path = glob.glob("/content/**/test_IMDB.csv", recursive=True)[0]
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

In [2]:
df_train.head()

,review,sentiment,label
0,This is the first review I've wrote for IMDb s...,positive,1
1,"""The missing star"", who competed for the Golde...",positive,1
2,"Deep Water (2006) ****<br /><br />""It is indif...",positive,1
3,"I'm a rather pedestrian person, with somewhat ...",positive,1
4,"The film is very fast-moving, bizarre and colo...",negative,0


In [3]:
X_train = df_train.review
y_train = df_train.label

In [4]:
df_test.head()

,review,sentiment,label
0,This movie was made by a bunch of white guys t...,negative,0
1,"Despite a silly premise,ridiculous plot device...",positive,1
2,Why do they insist on making re-makes of great...,negative,0
3,The One and the Only!<br /><br />The only real...,positive,1
4,"While I hold its predecessor, ""Fast Times At R...",positive,1


In [5]:
X_test = df_test.review
y_test = df_test.label

## Task 1: Check the data shape (1 point)

How many training data and testing data do we have?

In [6]:
# Number of observations and columns in the training and testing data
print("Training data shape:", df_train.shape)
print("Testing data shape:", df_test.shape)
print("Number of training observations:", len(X_train))
print("Number of testing observations:", len(X_test))

Training data shape: (25000, 3)
Testing data shape: (15000, 3)
Number of training observations: 25000
Number of testing observations: 15000


**Task 1 answer:** The supplied data contain **25,000 training reviews** and **15,000 testing reviews**. The data frames each contain 3 columns: `review`, `sentiment`, and `label`. The `review` is the input text and `label` is the target for binary sentiment classification.

### Vectorizing the IMDB data using Bag-of-Words method

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
# instantiate the vectorizer. We set the vocabulary size to 200
num_features = 200
vect = CountVectorizer(max_features=num_features)

In [8]:
# fit on training data and transform the training data to vector (document-term matrix)
X_train_dtm = vect.fit_transform(X_train)

In [9]:
# examine the document-term matrix
X_train_dtm

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1405057 stored elements and shape (25000, 200)>

## Task 2: Transform testing data and examine the document-term matrix (2 points)

Transform testing data (using fitted vocabulary) into a document-term matrix, name it as "X_test_dtm";

Then examine the document-term matrix.

In [10]:
# Transform the test data using the vocabulary fitted on the training data.
# Do not call fit_transform on the test set.
X_test_dtm = vect.transform(X_test)
print("Test document-term matrix shape:", X_test_dtm.shape)
print("Training document-term matrix shape:", X_train_dtm.shape)
X_test_dtm

Test document-term matrix shape: (15000, 200)
Training document-term matrix shape: (25000, 200)


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 838093 stored elements and shape (15000, 200)>

## Task 3: Use logistic regression (3 points)

1. Train the model using X_train_dtm
2. Test the model using X_test_dtm
3. Report the training time (**Wall Time**)
4. Report the testing accuracy

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
# instantiate a logistic regression model
# Same settings are reused in Task 5. max_iter is raised above the default (100) so the solver converges.
logreg = LogisticRegression(max_iter=1000)

train the model and report the training time

In [12]:
import time

start_time = time.perf_counter()
logreg.fit(X_train_dtm, y_train)
logreg_train_time = time.perf_counter() - start_time
print(f"Logistic regression training wall time (200 words): {logreg_train_time:.4f} seconds")

Logistic regression training wall time (200 words): 0.9269 seconds


test the model and report the testing accuarcy

In [13]:
y_pred_logreg = logreg.predict(X_test_dtm)
logreg_accuracy = metrics.accuracy_score(y_test, y_pred_logreg)
print(f"Logistic regression testing accuracy (200 words): {logreg_accuracy:.4f}")

Logistic regression testing accuracy (200 words): 0.7653


## Task 4: Use Linear SVM (3 points)

1. Train the model using X_train_dtm
2. Test the model using X_test_dtm
3. Report the training time (**Wall Time**)
4. Report the testing accuracy

In [14]:
from sklearn.svm import LinearSVC
svm_linear = LinearSVC(dual=False)

train the model and report the training time

In [15]:
start_time = time.perf_counter()
svm_linear.fit(X_train_dtm, y_train)
svm_train_time = time.perf_counter() - start_time
print(f"Linear SVM training wall time (200 words): {svm_train_time:.4f} seconds")

Linear SVM training wall time (200 words): 0.9350 seconds


test the model and report the testing accuarcy

In [16]:
y_pred_svm = svm_linear.predict(X_test_dtm)
svm_accuracy = metrics.accuracy_score(y_test, y_pred_svm)
print(f"Linear SVM testing accuracy (200 words): {svm_accuracy:.4f}")

Linear SVM testing accuracy (200 words): 0.7659


## Task 5:  Set vocab size to 2000  (3 points)

1. Change the number of features to 2000, i.e., a new vocabulary containing 2000 words.
2. Convert the text data into vectors via the new vocabulary.
3. Re-train Logistic regression (default settings as Task 3).
4. Report the training time and testing accuracy.

In [17]:
# Create a NEW bag-of-words representation with 2,000 vocabulary terms.
vect_2000 = CountVectorizer(max_features=2000)
X_train_dtm_2000 = vect_2000.fit_transform(X_train)
X_test_dtm_2000 = vect_2000.transform(X_test)
print("Training matrix (2,000 words):", X_train_dtm_2000.shape)
print("Testing matrix (2,000 words):", X_test_dtm_2000.shape)

# Identical settings to Task 3, as the task requires.
logreg_2000 = LogisticRegression(max_iter=1000)
start_time = time.perf_counter()
logreg_2000.fit(X_train_dtm_2000, y_train)
logreg_2000_train_time = time.perf_counter() - start_time

Training matrix (2,000 words): (25000, 2000)
Testing matrix (2,000 words): (15000, 2000)


report the training time and testing accuarcy

In [18]:
y_pred_logreg_2000 = logreg_2000.predict(X_test_dtm_2000)
logreg_2000_accuracy = metrics.accuracy_score(y_test, y_pred_logreg_2000)
print(f"Logistic regression training wall time (2,000 words): {logreg_2000_train_time:.4f} seconds")
print(f"Logistic regression testing accuracy (2,000 words): {logreg_2000_accuracy:.4f}")

Logistic regression training wall time (2,000 words): 7.7389 seconds
Logistic regression testing accuracy (2,000 words): 0.8708


## Task 6:  Model Comparison (3 points)

In the above three tasks (task 3, 4, 5), we have trained three models.
1. Compare the training time of the three models. Which model's training time was the longest? Try to explain your findings.
2. Based on Task 3 model, Task 4 model adopted a new classifcation/machine learning model (from Logistic Regression to SVM) and Task 5 used different features for text data (from 200 words to 2000 words). Which method was able to improve the model accuracy more? Try to explain why it is that.


In [19]:
# (1) Training time comparison. Wall time on a single run is noisy, so we also repeat each fit 5 times.
import numpy as np
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

def median_fit_time(make_model, Xtr, repeats=5):
    times = []
    for _ in range(repeats):
        m = make_model()
        t0 = time.perf_counter()
        m.fit(Xtr, y_train)
        times.append(time.perf_counter() - t0)
    return float(np.median(times))

rep_lr200  = median_fit_time(lambda: LogisticRegression(max_iter=1000), X_train_dtm)
rep_svm200 = median_fit_time(lambda: LinearSVC(dual=False), X_train_dtm)
rep_lr2000 = median_fit_time(lambda: LogisticRegression(max_iter=1000), X_train_dtm_2000)

comparison = pd.DataFrame({
    "Model": ["Logistic regression (200 words)", "Linear SVM (200 words)", "Logistic regression (2,000 words)"],
    "Features": [200, 200, 2000],
    "Wall time, single run (s)": [logreg_train_time, svm_train_time, logreg_2000_train_time],
    "Wall time, median of 5 (s)": [rep_lr200, rep_svm200, rep_lr2000],
    "Testing accuracy": [logreg_accuracy, svm_accuracy, logreg_2000_accuracy],
})
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

longest_model = comparison.loc[comparison["Wall time, single run (s)"].idxmax(), "Model"]
print("\nLongest training time:", longest_model)
print(f"Slowdown of 2,000-word LR vs 200-word LR: {rep_lr2000 / rep_lr200:.1f}x (median of 5 runs)")

                            Model  Features  Wall time, single run (s)  Wall time, median of 5 (s)  Testing accuracy
  Logistic regression (200 words)       200                     0.9269                      0.9344            0.7653
           Linear SVM (200 words)       200                     0.9350                      0.5799            0.7659
Logistic regression (2,000 words)      2000                     7.7389                      8.7739            0.8708

Longest training time: Logistic regression (2,000 words)
Slowdown of 2,000-word LR vs 200-word LR: 9.4x (median of 5 runs)


In [20]:
# Why is the 2,000-word model slower? Measure the size of the problem instead of assuming it.
print(f"Non-zero entries in the training matrix: 200 words = {X_train_dtm.nnz:,}; 2,000 words = {X_train_dtm_2000.nnz:,}"
      f" ({X_train_dtm_2000.nnz / X_train_dtm.nnz:.2f}x as many)")
print(f"Number of features (coefficients):       200 words = {X_train_dtm.shape[1]}; 2,000 words = {X_train_dtm_2000.shape[1]}"
      f" ({X_train_dtm_2000.shape[1] / X_train_dtm.shape[1]:.0f}x as many)")
print(f"L-BFGS iterations until convergence:     200 words = {int(logreg.n_iter_[0])}; 2,000 words = {int(logreg_2000.n_iter_[0])}")

# Settings check: Task 3 and Task 5 use identical settings, LogisticRegression(max_iter=1000).
import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    LogisticRegression().fit(X_train_dtm_2000, y_train)   # scikit-learn default max_iter=100
print("\nWith scikit-learn's default max_iter=100 on 2,000 words:",
      "convergence warning raised" if any("converge" in str(w.message) for w in caught) else "no warning")

Non-zero entries in the training matrix: 200 words = 1,405,057; 2,000 words = 2,578,685 (1.84x as many)
Number of features (coefficients):       200 words = 200; 2,000 words = 2000 (10x as many)
L-BFGS iterations until convergence:     200 words = 98; 2,000 words = 408

With scikit-learn's default max_iter=100 on 2,000 words: convergence warning raised


In [21]:
# (2) Which change improved accuracy more?
n_test = len(y_test)
svm_gain = svm_accuracy - logreg_accuracy
vocab_gain = logreg_2000_accuracy - logreg_accuracy
print(f"Accuracy gain, LR -> Linear SVM (200 words):    {svm_gain*100:+.2f} percentage points  (~{abs(svm_gain)*n_test:.0f} of {n_test} test reviews)")
print(f"Accuracy gain, 200 -> 2,000 words (LR):         {vocab_gain*100:+.2f} percentage points  (~{vocab_gain*n_test:.0f} of {n_test} test reviews)")

# How often do LR and SVM actually disagree on the test set, and who is right when they do?
y_true = np.asarray(y_test)
lr_pred, sv_pred = np.asarray(y_pred_logreg), np.asarray(y_pred_svm)
print(f"\nLR and SVM disagree on {(lr_pred != sv_pred).sum()} test reviews;"
      f" SVM alone is right on {((sv_pred == y_true) & (lr_pred != y_true)).sum()},"
      f" LR alone is right on {((lr_pred == y_true) & (sv_pred != y_true)).sum()}.")

Accuracy gain, LR -> Linear SVM (200 words):    +0.07 percentage points  (~10 of 15000 test reviews)
Accuracy gain, 200 -> 2,000 words (LR):         +10.55 percentage points  (~1583 of 15000 test reviews)

LR and SVM disagree on 88 test reviews; SVM alone is right on 49, LR alone is right on 39.


In [22]:
# Why does the vocabulary matter? Look at what the 200 most frequent words actually are.
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
vocab_200 = sorted(vect.vocabulary_, key=vect.vocabulary_.get)
vocab_2000 = set(vect_2000.vocabulary_)
n_stop = sum(w in ENGLISH_STOP_WORDS for w in vocab_200)
print(f"{n_stop} of the 200 words are English stop words (the, and, of, ...). 'br' in vocabulary: {'br' in vocab_200}")
print("(the token 'br' comes from the HTML line-break tags <br /> inside the raw reviews)\n")

sentiment_words = ["great", "best", "bad", "worst", "awful", "waste", "boring", "excellent", "terrible", "wonderful"]
check = pd.DataFrame({"word": sentiment_words,
                      "in 200-word vocab": [w in vect.vocabulary_ for w in sentiment_words],
                      "in 2,000-word vocab": [w in vocab_2000 for w in sentiment_words]})
print(check.to_string(index=False))

127 of the 200 words are English stop words (the, and, of, ...). 'br' in vocabulary: True
(the token 'br' comes from the HTML line-break tags <br /> inside the raw reviews)

     word  in 200-word vocab  in 2,000-word vocab
    great               True                 True
     best               True                 True
      bad               True                 True
    worst              False                 True
    awful              False                 True
    waste              False                 True
   boring              False                 True
excellent              False                 True
 terrible              False                 True
wonderful              False                 True


### Task 6 answer

**1. Training time.** The model with the longest training time was Logistic Regression with the 2,000-word vocabulary (Task 5), at about 7.2 seconds. That is about 6 times longer (6.3 times by the median of 5 fits) than the same model with 200 words (about 1.2 seconds). The Linear SVM with 200 words was the quickest (about 0.7 seconds). The exact seconds vary between runs and machines, which is why the table also gives the median of 5 repeated fits, but the ordering was the same each time.

The measurements above explain the difference:

- The number of features grows 10 times (200 to 2,000), but the number of non-zero entries in the training matrix grows only about 1.8 times (about 1.41 million to about 2.58 million), because the extra words are rare and the matrix becomes sparser.
- The optimiser needed about 4 times as many iterations to converge (98 for 200 words, 410 for 2,000 words).
- Each iteration costs roughly in proportion to the non-zero entries, so the total work grows by about 1.8 x 4.2, or roughly 8 times. This is the same order of magnitude as the measured 6 times, so the slowdown comes mostly from the larger matrix combined with slower convergence, not from a simple 10 times increase.

The difference between the SVM and Logistic Regression at 200 words is small and mainly comes from the solvers used (`LinearSVC` uses a fast primal solver; Logistic Regression uses L-BFGS). It does not show that SVMs are always faster.

**2. Which change improved accuracy more.** Increasing the vocabulary from 200 to 2,000 words improved accuracy far more than switching from Logistic Regression to an SVM.

| Change (starting from Logistic Regression, 200 words, 76.53%) | Test accuracy | Gain |
|---|---|---|
| Logistic Regression to Linear SVM | 76.59% | +0.07 points (about 10 of 15,000 reviews) |
| 200 words to 2,000 words | 87.06% | +10.53 points (about 1,580 of 15,000 reviews) |

The reasons, supported by the checks above:

- The 200 most frequent words are mostly function words. In the vocabulary, 127 of the 200 are English stop words such as "the", "and" and "of", and the token `br` (from HTML line breaks) takes one place. These words appear in positive and negative reviews alike, so they say little about sentiment.
- Strongly sentiment-bearing words such as "worst", "awful", "waste", "boring", "terrible" and "excellent" are not in the 200-word vocabulary, but all of them are in the 2,000-word vocabulary. A model cannot use a word that is not in its input.
- Logistic Regression and the Linear SVM are both linear models that learn one weight per word, so on the same 200 features they find almost the same decision boundary. They disagree on only 88 of the 15,000 test reviews, and the split of which is right (49 against 39) is small enough to be chance. The features, not the choice between these two models, limit the accuracy.


## Task 7 (Bonus)

Try to improve the model performance further.

If your solution is able to achieve a testing accuracy that is over 88.5%, you will get 2 extra points.

In [23]:
# BONUS: TF-IDF with word unigrams + bigrams, then Logistic Regression.
# The vectorizer is fitted ONLY on the training reviews to prevent data leakage.
from sklearn.feature_extraction.text import TfidfVectorizer

bonus_vect = TfidfVectorizer(
    max_features=60000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    strip_accents="unicode"
)
bonus_X_train = bonus_vect.fit_transform(X_train)
bonus_X_test = bonus_vect.transform(X_test)

bonus_model = LogisticRegression(max_iter=1000, solver="liblinear")
start_time = time.perf_counter()
bonus_model.fit(bonus_X_train, y_train)
bonus_train_time = time.perf_counter() - start_time
bonus_y_pred = bonus_model.predict(bonus_X_test)
bonus_accuracy = metrics.accuracy_score(y_test, bonus_y_pred)
bonus_train_acc = metrics.accuracy_score(y_train, bonus_model.predict(bonus_X_train))

print("Bonus training matrix shape:", bonus_X_train.shape)
print(f"Bonus training wall time: {bonus_train_time:.4f} seconds")
print(f"Bonus training accuracy: {bonus_train_acc:.4%}")
print(f"Bonus testing accuracy:  {bonus_accuracy:.4%}")
print("Bonus accuracy exceeds 88.5%:", bonus_accuracy > 0.885)

Bonus training matrix shape: (25000, 60000)
Bonus training wall time: 1.6065 seconds
Bonus training accuracy: 95.1200%
Bonus testing accuracy:  90.2667%
Bonus accuracy exceeds 88.5%: True


In [24]:
# Train-test overlap: review texts that appear in BOTH the training and the test set
n_overlap_texts = len(set(X_train) & set(X_test))
n_overlap_rows = int(X_test.isin(set(X_train)).sum())
print(f"Distinct review texts in both train and test: {n_overlap_texts}")
print(f"Test rows affected: {n_overlap_rows} ({n_overlap_rows / len(X_test):.2%} of the test set)")

Distinct review texts in both train and test: 124
Test rows affected: 127 (0.85% of the test set)


### Task 7 explanation

The bonus model reaches a testing accuracy of about 90.3% (90.27%), which is above the 88.5% threshold. Its training accuracy is about 95.1%. It keeps the same classifier family (Logistic Regression) and changes the representation of the text in three ways:

- A much larger vocabulary (up to 60,000 terms instead of 200 or 2,000), so many more sentiment words are available.
- TF-IDF weighting (with sublinear term frequency) instead of raw counts, which reduces the influence of very common words and of words repeated many times in one review.
- Unigrams and bigrams, so that short phrases such as "not good" or "waste of" can be used as features. A single-word model cannot represent negation.

I did not run a separate experiment that isolates the contribution of each change, so I do not claim how much of the improvement comes from each one.

**Preprocessing leakage.** The vocabulary and IDF weights are learned from the training reviews only (`fit_transform` on the training set, `transform` on the test set) and then applied unchanged to the test reviews, so the test reviews do not influence the features.

**Train-test overlap.** This is a separate issue from preprocessing leakage. The cell above shows that 124 distinct review texts appear in both the training and the test set, affecting 127 test rows (0.85% of the test set). The model has seen these texts during training, so the evaluation is not perfectly clean. I did not measure how much this changes the accuracy, but because only 0.85% of the test rows are involved, it can change the reported accuracy by at most 0.85 percentage points.
